#### Historical Replay, Root-Cause Analysis & Performance KPIs

**Objective**  
Build historical trip replay functionality, perform root-cause analysis on alerts, and calculate key performance metrics
- Accuracy of tracking reports
- Percentage of on-time deliveries tracked
- Reduction in route deviations

#### Path Setup & Imports



In [1]:
import sys
from pathlib import Path

notebook_dir = Path.cwd()
project_root = notebook_dir.parent
sys.path.append(str(project_root))

import pandas as pd
import numpy as np
import plotly.express as px

print("Libraries and path configured.")

Libraries and path configured.


#### Load Processed Data

In [2]:
df = pd.read_csv('../data/processed/noc_telematics_processed.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(f"Loaded {len(df):,} records with alerts")
print(f"Time range: {df['timestamp'].min()} → {df['timestamp'].max()}")

Loaded 4,260 records with alerts
Time range: 2025-04-01 06:00:00 → 2025-04-05 20:55:00


#### 1. Trip-Level Summary

In [4]:
trip_summary = df.groupby(['vehicle_id', 'trip_id', 'route']).agg({
    'timestamp': ['min', 'max'],
    'alert_type': lambda x: (x != 'Normal').sum(),
    'fuel_drop_alert': 'sum',
    'stop_duration_minutes': 'max',
    'route_deviation_km': 'max',
    'speed_kmh': 'mean'
}).reset_index()

trip_summary.columns = ['vehicle_id', 'trip_id', 'route', 'start_time', 'end_time', 
                       'total_alerts', 'fuel_theft_alerts', 'max_stop_min', 
                       'max_deviation_km', 'avg_speed_kmh']

trip_summary['trip_duration_hours'] = round((trip_summary['end_time'] - trip_summary['start_time']).dt.total_seconds() / 3600, 1)

print("Trip-level summary created.")
display(trip_summary.sort_values('total_alerts', ascending=False).head())

Trip-level summary created.


,vehicle_id,trip_id,route,start_time,end_time,total_alerts,fuel_theft_alerts,max_stop_min,max_deviation_km,avg_speed_kmh,trip_duration_hours
3,NOC-T001,NOC-T001_20250404,Dukem_to_Mekelle,2025-04-04 06:00:00,2025-04-05 04:25:00,28,0,35.0,0.72,42.748889,22.4
15,NOC-V501,NOC-V501_20250401,Dukem_to_Mekelle,2025-04-01 06:00:00,2025-04-02 04:25:00,20,1,0.0,0.38,41.211111,22.4
1,NOC-T001,NOC-T001_20250402,Dukem_to_Bahir_Dar,2025-04-02 06:00:00,2025-04-02 20:55:00,19,1,40.0,0.95,47.535000,14.9
2,NOC-T001,NOC-T001_20250403,Dukem_to_Dire_Dawa,2025-04-03 06:00:00,2025-04-03 18:25:00,15,0,40.0,0.13,44.395333,12.4
29,SUB-T102,SUB-T102_20250405,Dukem_to_Jimma,2025-04-05 06:00:00,2025-04-05 17:10:00,12,0,0.0,0.19,46.834074,11.2


#### 2. Historical Trip Replay Function

In [5]:
def replay_trip(vehicle_id, trip_id=None):
    if trip_id is None:
        trip_id = df[df['vehicle_id'] == vehicle_id]['trip_id'].iloc[0]
    
    trip_data = df[(df['vehicle_id'] == vehicle_id) & (df['trip_id'] == trip_id)].copy()
    trip_data = trip_data.sort_values('timestamp')
    
    print(f"Replaying Trip: {trip_id}")
    print(f"Vehicle: {vehicle_id} | Route: {trip_data['route'].iloc[0]}")
    print(f"Duration: {trip_data['timestamp'].max() - trip_data['timestamp'].min()}")
    print(f"Total Alerts: {(trip_data['alert_type'] != 'Normal').sum()}\n")
    
    alerts = trip_data[trip_data['alert_type'] != 'Normal']
    if not alerts.empty:
        print(" Alerts in this trip:")
        display(alerts[['timestamp', 'alert_type', 'alert_severity', 'speed_kmh', 
                       'stop_duration_minutes', 'fuel_level_percent', 'valve_status']].round(2))
    
    return trip_data

# Demo
print("Trip with Fuel Theft Suspect")
replay_trip('NOC-T001')

Trip with Fuel Theft Suspect
Replaying Trip: NOC-T001_20250401
Vehicle: NOC-T001 | Route: Dukem_to_Adama
Duration: 0 days 03:40:00
Total Alerts: 5

 Alerts in this trip:


C:\Users\tohiba\AppData\Local\Temp\ipykernel_288\4125322879.py:17: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  'stop_duration_minutes', 'fuel_level_percent', 'valve_status']].round(2))


,timestamp,alert_type,alert_severity,speed_kmh,stop_duration_minutes,fuel_level_percent,valve_status
21,2025-04-01 07:45:00,Valve Tampering,High,0.7,25.0,75.42,OPEN
22,2025-04-01 07:50:00,Valve Tampering,High,6.1,30.0,75.17,OPEN
23,2025-04-01 07:55:00,Prolonged Stop,High,4.9,35.0,74.86,CLOSED
24,2025-04-01 08:00:00,Prolonged Stop,High,0.4,40.0,74.24,CLOSED
38,2025-04-01 09:10:00,Harsh Driving,Medium,68.5,0.0,65.74,CLOSED


,timestamp,vehicle_id,vehicle_type,owner,trip_id,latitude,longitude,speed_kmh,odometer_km,fuel_level_percent,...,route,is_moving,harsh_braking,harsh_acceleration,device_status,stop_duration_minutes,route_deviation_km,fuel_drop_alert,alert_type,alert_severity
0,2025-04-01 06:00:00,NOC-T001,Fuel Tanker,NOC,NOC-T001_20250401,8.800370,38.922388,41.3,63800.7,86.64,...,Dukem_to_Adama,True,0,0,ONLINE,0.0,0.01,False,Normal,Low
1,2025-04-01 06:05:00,NOC-T001,Fuel Tanker,NOC,NOC-T001_20250401,8.791891,38.927145,51.7,63801.7,85.85,...,Dukem_to_Adama,True,0,0,ONLINE,0.0,0.05,False,Normal,Low
2,2025-04-01 06:10:00,NOC-T001,Fuel Tanker,NOC,NOC-T001_20250401,8.794186,38.926773,56.0,63802.0,85.15,...,Dukem_to_Adama,True,0,0,ONLINE,0.0,0.03,False,Normal,Low
3,2025-04-01 06:15:00,NOC-T001,Fuel Tanker,NOC,NOC-T001_20250401,8.792310,38.933654,52.3,63802.8,84.33,...,Dukem_to_Adama,True,0,0,ONLINE,0.0,0.00,False,Normal,Low
4,2025-04-01 06:20:00,NOC-T001,Fuel Tanker,NOC,NOC-T001_20250401,8.782612,38.962432,75.9,63806.1,83.63,...,Dukem_to_Adama,True,0,0,ONLINE,0.0,0.03,False,Normal,Low
5,2025-04-01 06:25:00,NOC-T001,Fuel Tanker,NOC,NOC-T001_20250401,8.775344,38.963089,68.7,63806.9,82.86,...,Dukem_to_Adama,True,0,0,ONLINE,0.0,0.00,False,Normal,Low
6,2025-04-01 06:30:00,NOC-T001,Fuel Tanker,NOC,NOC-T001_20250401,8.775635,38.970122,49.2,63807.7,82.07,...,Dukem_to_Adama,True,0,0,ONLINE,0.0,0.02,False,Normal,Low
7,2025-04-01 06:35:00,NOC-T001,Fuel Tanker,NOC,NOC-T001_20250401,8.750803,38.983238,63.8,63810.8,81.41,...,Dukem_to_Adama,True,0,0,ONLINE,0.0,0.01,False,Normal,Low
8,2025-04-01 06:40:00,NOC-T001,Fuel Tanker,NOC,NOC-T001_20250401,8.747773,38.985938,62.6,63811.2,80.69,...,Dukem_to_Adama,True,0,0,OFFLINE,0.0,0.02,False,Normal,Low
9,2025-04-01 06:45:00,NOC-T001,Fuel Tanker,NOC,NOC-T001_20250401,8.750118,38.993669,76.2,63812.1,80.04,...,Dukem_to_Adama,True,0,0,ONLINE,0.0,0.00,False,Normal,Low


#### 3. Performance Metrics

In [6]:
total_records = len(df)
alerted_records = (df['alert_type'] != 'Normal').sum()
tracking_accuracy = round(100 * (1 - alerted_records / total_records), 2)

total_trips = trip_summary.shape[0]
on_time_trips = trip_summary[trip_summary['max_deviation_km'] <= 6.0].shape[0]
on_time_pct = round((on_time_trips / total_trips) * 100, 1)

print("NOC GPS MONITORING KPIs")
print("=" * 45)
print(f"Tracking Report Accuracy       : {tracking_accuracy}%")
print(f"On-Time Deliveries Tracked     : {on_time_pct}%")
print(f"Average Max Route Deviation    : {trip_summary['max_deviation_km'].mean():.2f} km")
print(f"Total Alerts Generated         : {alerted_records}")
print(f"Fuel Theft Suspect Alerts      : {df['fuel_drop_alert'].sum()}")
print(f"Valve Tampering Events         : {(df['valve_status'] == 'OPEN').sum()}")

NOC GPS MONITORING KPIs
Tracking Report Accuracy       : 93.66%
On-Time Deliveries Tracked     : 100.0%
Average Max Route Deviation    : 0.89 km
Total Alerts Generated         : 270
Fuel Theft Suspect Alerts      : 8
Valve Tampering Events         : 42


#### 4. Root-Cause Analysis & Recommendations to NOC

In [7]:
print("Top Vehicles by Alert Count:")
print(df[df['alert_type'] != 'Normal']['vehicle_id'].value_counts().head(6))

print("\nMost Frequent High-Severity Alerts:")
high_alerts = df[df['alert_severity'] == 'High']['alert_type'].value_counts()
print(high_alerts)

Top Vehicles by Alert Count:
vehicle_id
NOC-T001    74
NOC-V501    47
SUB-T102    45
SUB-T101    39
NOC-T002    38
NOC-T003    27
Name: count, dtype: int64

Most Frequent High-Severity Alerts:
alert_type
Prolonged Stop        38
Valve Tampering       35
Fuel Theft Suspect     7
Name: count, dtype: int64


#### Save Final Data

In [9]:
import os
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/noc_telematics_with_alerts_final.csv', index=False)
trip_summary.to_csv('../data/processed/trip_summary.csv', index=False)

print("Final processed data and trip summary saved.")
print("Completed successfully")

Final processed data and trip summary saved.
Completed successfully
